# Hafta 11 · Varyasyonel Kuantum Algoritmaları: Hibrit Döngü, VQE Fikri ve QAOA
**Ders:** Kuantum Hesaplama ve Uygulamaları · **Lab süresi:** ~50 dk (+ ödev) · **Ortam:** Google Colab (CPU yeterli)

Bu hafta kuantum devresini **parametreli bir fonksiyon** gibi kullanıyoruz: parametreleri klasik bir optimizer ayarlıyor, kuantum devresi ise bir **maliyet (puan) fonksiyonunun** değerini hesaplıyor. Bu, makine öğrenmesindeki eğitim döngüsünün aynısıdır. Önce küçük bir matrisin en küçük özdeğerini bulacağız (VQE fikri), sonra **MaxCut** graf problemini **QAOA** ile çözeceğiz.

| Bölüm | Konu | Süre |
|---|---|---|
| 0 | Kurulum, yardımcılar, veri seti | 3 dk |
| A | Parametreli devreler: `Parameter`, `ParameterVector` | 5 dk |
| B | Maliyet = beklenen değer: Pauli dizileri, `SparsePauliOp`, sayımlar ve `StatevectorEstimator` | 8 dk |
| C | Türev: parameter-shift kuralı ve sonlu farklar | 7 dk |
| D | Optimizer'lar: gradyan inişi, COBYLA, SPSA, Adam | 6 dk |
| E | VQE fikri: bir matrisin en küçük özdeğeri | 7 dk |
| F | QAOA ile MaxCut: uçtan uca | 14 dk |
| G | Sınırlamalar ve gerçekçi beklentiler | okuma |
| H | Alıştırmalar (8 adet, `assert` ile kontrol) | ödev |

**Bit sırası:** Dersin tamamında olduğu gibi **Qiskit sırası** (q₀ en sağda). `SparsePauliOp("IZ")` → Z kapısı **q₀** üzerindedir.

## 0 · Kurulum

In [ ]:
!pip install -q qiskit qiskit-aer pylatexenc

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import product
from scipy.optimize import minimize
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import Parameter, ParameterVector
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.primitives import StatevectorEstimator, StatevectorSampler
from qiskit_aer import AerSimulator

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
NAVY, BLUE, ORANGE, GRAY = "#1F3A5F", "#2E6DB4", "#D9822B", "#8A94A6"
rng = np.random.default_rng(2026)
estimator = StatevectorEstimator()          # kesin (gürültüsüz) beklenen değer
aer = AerSimulator(seed_simulator=11)       # shot tabanlı ölçüm
print("hazır")

### Veri seti: MaxCut grafları
Aşağıdaki fonksiyon ders boyunca kullanılan `datasets.py` dosyasından **aynen** kopyalanmıştır (sabit tohum, her çalıştırmada aynı). Bu veri setinin CSV'si ayrıca verilmiştir: `maxcut_graflar.csv` (sütunlar: `graph_id, u, v, weight`).

In [ ]:
def ds_maxcut_graphs():
    """11. hafta QAOA için küçük graflar: graph_id, u, v, weight."""
    graphs = {
        "kare_4": [(0, 1), (1, 2), (2, 3), (3, 0)],
        "ucgen_kuyruk_4": [(0, 1), (1, 2), (2, 0), (2, 3)],
        "tam_4": [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)],
        "halka_5": [(0, 1), (1, 2), (2, 3), (3, 4), (4, 0)],
        "rastgele_6": [(0, 1), (0, 2), (1, 3), (2, 3), (2, 4), (3, 5), (4, 5), (1, 4)],
    }
    rows = [{"graph_id": g, "u": u, "v": v, "weight": 1.0} for g, es in graphs.items() for u, v in es]
    rng = np.random.default_rng(SEED)
    for u, v in [(0, 1), (0, 2), (1, 2), (1, 3), (2, 3), (3, 4), (2, 4)]:
        rows.append({"graph_id": "agirlikli_5", "u": u, "v": v, "weight": float(np.round(rng.uniform(0.5, 2.0), 2))})
    return pd.DataFrame(rows)

SEED = 42
df = ds_maxcut_graphs()

def load_graphs(df):
    """graph_id -> (düğüm sayısı n, [(u, v, w), ...])"""
    out = {}
    for g, d in df.groupby("graph_id", sort=False):
        e = [(int(u), int(v), float(w)) for u, v, w in zip(d.u, d.v, d.weight)]
        out[g] = (max(max(u, v) for u, v, _ in e) + 1, e)
    return out

GRAPHS = load_graphs(df)
for g, (n, e) in GRAPHS.items():
    print(f"{g:15s} n={n}  kenar={len(e)}  toplam ağırlık={sum(w for *_, w in e):.2f}")
df.head(8)

In [ ]:
def draw_graph(n, edges, bits=None, title=None, ax=None, show_w=False):
    """Grafı çember düzeninde çizer. bits (Qiskit sırası) verilirse: mavi = 1 kümesi, turuncu kesikli = kesilen kenar."""
    ax = ax or plt.gca()
    ang = np.pi/2 - 2*np.pi*np.arange(n)/n; P = np.stack([np.cos(ang), np.sin(ang)], 1)
    x = [int(c) for c in bits[::-1]] if bits else None
    for u, v, w in edges:
        cut = bits is not None and x[u] != x[v]
        ax.plot(*zip(P[u], P[v]), color=ORANGE if cut else GRAY, lw=3 if cut else 1.5, ls="--" if cut else "-", zorder=1)
        if show_w:
            m = (P[u] + P[v]) / 2; ax.text(*m, f"{w:g}", fontsize=8, ha="center", va="center", bbox=dict(fc="white", ec="none", pad=0.3))
    for i in range(n):
        on = bits is not None and x[i] == 1
        ax.add_patch(plt.Circle(P[i], 0.17, fc=BLUE if on else "white", ec=NAVY, lw=1.6, zorder=3))
        ax.text(*P[i], str(i), ha="center", va="center", color="white" if on else NAVY, fontweight="bold", zorder=4)
    ax.set_xlim(-1.35, 1.35); ax.set_ylim(-1.35, 1.35); ax.set_aspect("equal"); ax.axis("off")
    if title: ax.set_title(title, fontsize=10, color=NAVY)

fig, axs = plt.subplots(1, 6, figsize=(16, 3))
for ax, (g, (n, e)) in zip(axs, GRAPHS.items()):
    draw_graph(n, e, title=g, ax=ax, show_w=(g == "agirlikli_5"))
plt.show()

---
## A · Parametreli devreler (ansatz)
Bir **ansatz** (Almanca "yaklaşım/taslak"), içinde ayarlanabilir açılar bulunan bir devre şablonudur. Yazılımcı gözüyle: `def devre(theta): ...` şeklinde **parametreli bir fonksiyon**. Qiskit'te açıyı sayı yerine sembol olarak yazarız; devreyi bir kez kurar, farklı değerlerle defalarca çalıştırırız.

| Qiskit | Anlamı |
|---|---|
| `Parameter("θ")` | Tek bir sembolik parametre |
| `ParameterVector("θ", 4)` | θ[0], …, θ[3] |
| `qc.parameters` | Devredeki parametreler (**alfabetik sıralı!**) |
| `qc.assign_parameters([...])` | Değer atanmış yeni devre |

In [ ]:
theta = Parameter("θ")
qc1 = QuantumCircuit(1); qc1.ry(theta, 0)
display(qc1.draw("mpl"))

for t in [0, np.pi/3, 2*np.pi/3, np.pi]:
    sv = Statevector(qc1.assign_parameters([t]))
    print(f"θ = {t:.3f}  durum = {sv.data.real}  P(0) = {sv.probabilities()[0]:.3f}")

In [ ]:
# 2 kübitlik ansatz: Ry katmanı + CNOT + Ry katmanı (4 parametre)
th = ParameterVector("θ", 4)
ansatz = QuantumCircuit(2)
ansatz.ry(th[0], 0); ansatz.ry(th[1], 1)
ansatz.cx(0, 1)
ansatz.ry(th[2], 0); ansatz.ry(th[3], 1)
display(ansatz.draw("mpl"))
print("parametreler:", list(ansatz.parameters), " sayısı:", ansatz.num_parameters)
print("θ = [0.1, 0.2, 0.3, 0.4] için durum:", Statevector(ansatz.assign_parameters([0.1, 0.2, 0.3, 0.4])).data.real)

⚠️ **Dikkat:** `qc.parameters` alfabetik sıralıdır. QAOA'da `β` ve `γ` birlikte kullanıldığında sıra **β, γ** olur (β < γ). Değer listesi verirken bu sıraya uyun ya da sözlük (`{param: değer}`) kullanın.

---
## B · Maliyet = beklenen değer
Varyasyonel algoritmalarda optimize edilen sayı bir **beklenen değerdir**: her ölçüm sonucuna (bit dizisine) bir **puan** verilir, puanların olasılıkla ağırlıklı ortalaması alınır.

- **Z** (tek kübit): bit 0 → +1, bit 1 → −1. ⟨Z⟩ = P(0) − P(1) (2. hafta).
- **Z₀Z₁**: iki bit **aynıysa** +1, **farklıysa** −1 (eşlik / parity). ⟨ZZ⟩ = P(aynı) − P(farklı).
- **X₀**: X tabanında ölçülür → ölçümden önce q₀'a **H** uygulanır, sonra Z gibi okunur.

Qiskit'te bu puan fonksiyonları `SparsePauliOp` ile yazılır. Etiket stringinde **en sağdaki karakter q₀**'dır.

In [ ]:
ZZ = SparsePauliOp("ZZ")
Z0 = SparsePauliOp("IZ")            # Z, q0 üzerinde
X0 = SparsePauliOp("IX")            # X, q0 üzerinde
M = SparsePauliOp.from_list([("ZZ", 1.0), ("IX", 0.5), ("XI", 0.5)])   # bu haftanın matrisi
print(M)
print("ZZ matrisi köşegeni:", np.diag(ZZ.to_matrix()).real, " (|00>,|01>,|10>,|11> için puanlar)")
print("M (4x4):\n", M.to_matrix().real)

### Sayımlardan beklenen değer
Bir Pauli-Z dizisinin puanı = seçili bitlerin **eşliği**: tek sayıda 1 varsa −1, çift sayıda ise +1.

In [ ]:
def z_string_value(bits, zmask):
    """bits: '01' gibi (Qiskit sırası), zmask: 'ZZ', 'IZ' gibi. Seçili bitlerde 1 sayısı çiftse +1, tekse -1."""
    ones = sum(int(b) for b, p in zip(bits, zmask) if p == "Z")
    return 1 - 2 * (ones % 2)

def expval_counts(counts, zmask):
    N = sum(counts.values())
    return sum(c * z_string_value(k, zmask) for k, c in counts.items()) / N

# İki kübitli örnek durum: Ry(1.2) q0'a, Ry(0.5) q1'e, sonra CNOT
qc = QuantumCircuit(2); qc.ry(1.2, 0); qc.ry(0.5, 1); qc.cx(0, 1); qc.measure_all()
counts = aer.run(transpile(qc, aer), shots=4000).result().get_counts()
print("sayımlar:", counts)
qc_nom = qc.remove_final_measurements(inplace=False)
for mask in ["IZ", "ZI", "ZZ"]:
    exact = Statevector(qc_nom).expectation_value(SparsePauliOp(mask)).real
    print(f"<{mask}> sayımlardan = {expval_counts(counts, mask):+.4f}   kesin = {exact:+.4f}")

In [ ]:
# <X0> nasıl ölçülür? Ölçümden önce q0'a H uygula (taban değiştirme), sonra Z gibi oku
qc = QuantumCircuit(2); qc.ry(1.2, 0); qc.ry(0.5, 1); qc.cx(0, 1)
qx = qc.copy(); qx.h(0); qx.measure_all()
cx_counts = aer.run(transpile(qx, aer), shots=4000).result().get_counts()
print("<X0> sayımlardan:", expval_counts(cx_counts, "IZ"), "  kesin:", Statevector(qc).expectation_value(X0).real)

### `StatevectorEstimator`: beklenen değeri doğrudan hesaplayan primitive
Qiskit 2.x'te **Estimator** bir "PUB" (devre, gözlemlenebilir, parametre değerleri) listesi alır ve `result()[i].data.evs` içinde beklenen değerleri döndürür. Parametre değerleri **dizi** olarak verilirse tek çağrıda birçok nokta hesaplanır (broadcasting).

In [ ]:
job = estimator.run([(ansatz, M, [0.1, 0.2, 0.3, 0.4])])
print("<M> =", job.result()[0].data.evs)

# aynı devre, 3 farklı parametre vektörü tek çağrıda
vals = np.array([[0.1, 0.2, 0.3, 0.4], [0, 0, 0, 0], [np.pi, 0, 0, 0]])
print("<M> (3 nokta) =", estimator.run([(ansatz, M, vals)]).result()[0].data.evs)

def cost(x, circ=ansatz, op=M):
    """Varyasyonel maliyet fonksiyonu: C(θ) = <ψ(θ)|M|ψ(θ)>"""
    return float(estimator.run([(circ, op, x)]).result()[0].data.evs)
print("cost([0,0,0,0]) =", cost([0, 0, 0, 0]), "  (|00> için ZZ puanı +1)")

---
## C · Türev: parameter-shift kuralı
Gradyan tabanlı optimizer'lar ∂C/∂θ ister. Kuantum devresinde geri yayılım (backprop) yapamayız, çünkü ara durumlara erişemeyiz. Çözüm: devreyi **iki kez daha** çalıştırmak.

$$\frac{\partial C}{\partial \theta_k} = \frac{C(\theta_k + \pi/2) - C(\theta_k - \pi/2)}{2}$$

Bu kural Rx, Ry, Rz, RZZ gibi **dönme kapıları** için **kesindir** (yaklaşık değil). Sonlu fark ise yaklaşıktır ve shot gürültüsüne çok duyarlıdır.

In [ ]:
def f(t):                                  # tek kübit: Ry(t)|0>, ölç Z  -> cos t
    return cost([t], qc1, SparsePauliOp("Z"))

t0 = 0.8
ps = (f(t0 + np.pi/2) - f(t0 - np.pi/2)) / 2
print(f"parameter-shift  : {ps:.10f}")
print(f"analitik -sin(t) : {-np.sin(t0):.10f}")
rows = []
for h in [1e-1, 1e-2, 1e-3, 1e-4]:
    fd = (f(t0 + h) - f(t0 - h)) / (2*h)
    rows.append({"h": h, "sonlu fark": fd, "hata": abs(fd + np.sin(t0))})
pd.DataFrame(rows)

In [ ]:
# Shot gürültüsü varken: 1000 shot ile türev tahmini
def f_shots(t, shots=1000):
    q = QuantumCircuit(1); q.ry(t, 0); q.measure_all()
    c = aer.run(transpile(q, aer), shots=shots, seed_simulator=int(rng.integers(1e9))).result().get_counts()   # her çağrıda yeni tohum
    return (c.get("0", 0) - c.get("1", 0)) / shots

ps_est = [(f_shots(t0 + np.pi/2) - f_shots(t0 - np.pi/2)) / 2 for _ in range(20)]
fd_est = [(f_shots(t0 + 0.01) - f_shots(t0 - 0.01)) / 0.02 for _ in range(20)]
print(f"gerçek türev              : {-np.sin(t0):+.3f}")
print(f"parameter-shift (1000 shot): ort {np.mean(ps_est):+.3f}, std {np.std(ps_est):.3f}")
print(f"sonlu fark h=0.01          : ort {np.mean(fd_est):+.3f}, std {np.std(fd_est):.3f}   <- çok gürültülü!")

In [ ]:
def ps_gradient(fun, x):
    """Çok parametreli parameter-shift gradyanı: her parametre için 2 değerlendirme."""
    x = np.asarray(x, float); g = np.zeros_like(x)
    for k in range(len(x)):
        e = np.zeros_like(x); e[k] = np.pi/2
        g[k] = (fun(x + e) - fun(x - e)) / 2
    return g

x = np.array([0.1, 0.2, 0.3, 0.4])
g_ps = ps_gradient(cost, x)
g_fd = np.array([(cost(x + 1e-5*np.eye(4)[k]) - cost(x - 1e-5*np.eye(4)[k])) / 2e-5 for k in range(4)])
print("parameter-shift:", g_ps)
print("sonlu fark     :", g_fd)
print("uyumlu mu?", np.allclose(g_ps, g_fd, atol=1e-6))

---
## D · Optimizer'lar
| Optimizer | Türev ister mi? | Adım başına devre çalıştırma (P parametre) | Not |
|---|---|---|---|
| Gradyan inişi | Evet (parameter-shift) | 2P | En basit; öğrenme oranı η seçimi önemli |
| Adam | Evet | 2P | ML'deki Adam'ın aynısı; momentum + uyarlamalı adım |
| COBYLA | Hayır | ~1 | `scipy.optimize.minimize(method="COBYLA")`; küçük problemlerde çok iyi |
| SPSA | Hayır (tahmin eder) | **2** (P'den bağımsız!) | Gürültülü donanım için tasarlanmış |

In [ ]:
def gradient_descent(fun, x0, eta=0.2, steps=60):
    x = np.array(x0, float); hist = [fun(x)]
    for _ in range(steps):
        x = x - eta * ps_gradient(fun, x); hist.append(fun(x))
    return x, hist

def adam(fun, x0, lr=0.2, steps=60, b1=0.9, b2=0.999):
    x = np.array(x0, float); m = np.zeros_like(x); v = np.zeros_like(x); hist = [fun(x)]
    for k in range(1, steps + 1):
        g = ps_gradient(fun, x); m = b1*m + (1-b1)*g; v = b2*v + (1-b2)*g*g
        x = x - lr * (m/(1-b1**k)) / (np.sqrt(v/(1-b2**k)) + 1e-8); hist.append(fun(x))
    return x, hist

def spsa(fun, x0, steps=150, a=0.3, c=0.2, seed=3):
    r = np.random.default_rng(seed); x = np.array(x0, float); hist = [fun(x)]
    for k in range(1, steps + 1):
        ak, ck = a / k**0.602, c / k**0.101
        d = r.choice([-1, 1], len(x))                       # tüm parametreleri aynı anda rastgele it
        g = (fun(x + ck*d) - fun(x - ck*d)) / (2*ck) * d
        x = x - ak * g; hist.append(fun(x))
    return x, hist

x0 = [0.1, 0.2, 0.3, 0.4]
_, h_gd = gradient_descent(cost, x0)
_, h_adam = adam(cost, x0)
_, h_spsa = spsa(cost, x0)
h_cob = []; res = minimize(lambda x: (h_cob.append(cost(x)), h_cob[-1])[1], x0, method="COBYLA", options={"maxiter": 150})
print(f"COBYLA: {res.fun:.5f} ({res.nfev} değerlendirme)")
print(f"GD: {h_gd[-1]:.5f}   Adam: {h_adam[-1]:.5f}   SPSA: {h_spsa[-1]:.5f}")

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(np.arange(len(h_gd)) * 8, h_gd, label="gradyan inişi (8 değ./adım)", color=GRAY)
plt.plot(np.arange(len(h_adam)) * 8, h_adam, label="Adam (8 değ./adım)", color=BLUE)
plt.plot(np.arange(len(h_spsa)) * 2, h_spsa, label="SPSA (2 değ./adım)", color="#999999", ls=":")
plt.plot(h_cob, label="COBYLA", color=NAVY)
plt.axhline(-np.sqrt(2), color=ORANGE, ls="--", label="kesin: -√2")
plt.xlabel("maliyet değerlendirme sayısı"); plt.ylabel("C(θ)"); plt.legend(); plt.xlim(0, 500); plt.title("Optimizer karşılaştırması")
plt.show()

---
## E · VQE fikri: bir matrisin en küçük özdeğeri
**Problem:** Simetrik bir M matrisinin **en küçük özdeğerini** bul. Klasik yol: `np.linalg.eigvalsh(M)` — ama M 2ⁿ × 2ⁿ boyutunda ve n büyüdükçe bu imkânsızlaşır.

**VQE (Variational Quantum Eigensolver) fikri:** Her durum için ⟨ψ|M|ψ⟩ ≥ λ_min (varyasyon ilkesi — Rayleigh bölümü). Öyleyse ⟨ψ(θ)|M|ψ(θ)⟩'yi θ üzerinden **minimize edersek** λ_min'e yukarıdan yaklaşırız. Bu haftaki matris: **M = Z₀Z₁ + 0.5·X₀ + 0.5·X₁**.

In [ ]:
Mm = M.to_matrix().real
ev = np.linalg.eigvalsh(Mm)
print("özdeğerler:", ev, "  en küçük =", ev[0], " = -√2 =", -np.sqrt(2))

# Rayleigh sınırı: rastgele durumlar hiçbir zaman λ_min'in altına inemez
rand_vals = []
for _ in range(2000):
    v = rng.normal(size=4) + 1j*rng.normal(size=4); v /= np.linalg.norm(v)
    rand_vals.append(np.vdot(v, Mm @ v).real)
print("2000 rastgele durumda en küçük <M>:", min(rand_vals), " >= λ_min ✓")

In [ ]:
# Tek kübit ısınma: M1 = Z + X, ansatz Ry(θ)|0>  -> C(θ) = cos θ + sin θ
M1 = SparsePauliOp.from_list([("Z", 1), ("X", 1)])
ts = np.linspace(0, 2*np.pi, 200)
land = estimator.run([(qc1, M1, ts.reshape(-1, 1))]).result()[0].data.evs
plt.figure(figsize=(8, 3.5)); plt.plot(ts, land, color=NAVY, label="C(θ)")
plt.axhline(np.linalg.eigvalsh(M1.to_matrix())[0], color=ORANGE, ls="--", label="λ_min = -√2")
plt.xlabel("θ"); plt.legend(); plt.title("Tek kübit maliyet manzarası"); plt.show()
print("manzaranın minimumu:", land.min(), " θ* ≈", ts[np.argmin(land)], " (5π/4 =", 5*np.pi/4, ")")

In [ ]:
# 2 kübit VQE: birkaç rastgele başlangıç + COBYLA (yerel minimumlara karşı)
best = None
for s in range(4):
    x0 = np.random.default_rng(s).uniform(0, 2*np.pi, 4)
    r = minimize(cost, x0, method="COBYLA", options={"maxiter": 300})
    print(f"başlangıç {s}: C = {r.fun:.6f}  ({r.nfev} değerlendirme)")
    if best is None or r.fun < best.fun: best = r
print(f"\nVQE sonucu: {best.fun:.6f}   kesin (eigvalsh): {ev[0]:.6f}   fark: {best.fun - ev[0]:.1e}")
psi = Statevector(ansatz.assign_parameters(best.x))
print("bulunan durum:", np.round(psi.data.real, 4))
print("olasılıklar  :", dict(zip(["00","01","10","11"], np.round(psi.probabilities(), 3))))

---
## F · QAOA ile MaxCut
**MaxCut:** Grafın düğümlerini iki kümeye ayır; iki ucu **farklı** kümelerde olan kenarlar "kesilir". Amaç kesilen kenar ağırlıkları toplamını **en büyük** yapmak. Her düğüm bir kübit; bir bit dizisi (bitstring) bir bölmedir.

$$\text{kesim}(x) = \sum_{(i,j)\in E} w_{ij}\,[x_i \neq x_j] \qquad\Longleftrightarrow\qquad C = \sum_{(i,j)} w_{ij}\,\frac{1 - Z_iZ_j}{2}$$

Çünkü ZᵢZⱼ bitler aynıyken +1 (→ katkı 0), farklıyken −1 (→ katkı w). Yani **maliyet operatörü, her bitstring'e onun kesim değerini puan olarak veren köşegen bir matristir.**

In [ ]:
def cut_value(bits, edges):
    """bits Qiskit sırası: en sağdaki karakter düğüm 0."""
    x = [int(c) for c in bits[::-1]]
    return sum(w for u, v, w in edges if x[u] != x[v])

def brute_force(n, edges):
    vals = {format(k, f"0{n}b"): cut_value(format(k, f"0{n}b"), edges) for k in range(2**n)}
    best = max(vals.values())
    return best, [b for b, v in vals.items() if np.isclose(v, best)], vals

n, E = GRAPHS["kare_4"]
best, opts, vals = brute_force(n, E)
print("kare_4 en iyi kesim:", best, " çözümler:", opts)
fig, axs = plt.subplots(1, 4, figsize=(12, 3))
for ax, b in zip(axs, ["0000", "0001", "0011", "0101"]):
    draw_graph(n, E, b, title=f"{b}: kesim = {cut_value(b, E):g}", ax=ax)
plt.show()

In [ ]:
def maxcut_op(n, edges):
    """C = Σ w (1 - Z_i Z_j)/2  ->  SparsePauliOp"""
    terms = [("ZZ", [u, v], -w/2) for u, v, w in edges]
    const = sum(w for *_, w in edges) / 2
    return SparsePauliOp.from_sparse_list(terms + [("", [], const)], num_qubits=n).simplify()

C_op = maxcut_op(n, E)
print(C_op)
diag = np.diag(C_op.to_matrix()).real
print("köşegen (her bitstring'in puanı):", diag)
print("kaba kuvvet ile aynı mı?", np.allclose(diag, [vals[format(k, f'0{n}b')] for k in range(2**n)]))

### QAOA devresi
1. **H katmanı:** tüm bölmelerin eşit süperpozisyonu.
2. **Maliyet katmanı U_C(γ):** her kenar için `RZZ(2γ·w)` — kesilen/kesilmeyen bölmelere farklı **faz** verir.
3. **Karıştırıcı U_M(β):** her kübite `RX(2β)` — fazları olasılık farkına çevirir (Grover'daki difüzyona benzer rol).
4. p kez tekrar, sonra ölçüm. Toplam **2p** parametre.

In [ ]:
def qaoa_circuit(n, edges, p):
    gam = ParameterVector("γ", p); bet = ParameterVector("β", p)
    qc = QuantumCircuit(n); qc.h(range(n))
    for k in range(p):
        for u, v, w in edges:
            qc.rzz(2 * gam[k] * w, u, v)
        qc.rx(2 * bet[k], range(n))
    return qc

qc_p1 = qaoa_circuit(n, E, 1)
display(qc_p1.draw("mpl", fold=-1))
print("parametre sırası:", list(qc_p1.parameters), " <- dikkat: β önce, γ sonra (alfabetik)")

In [ ]:
# Tek kenar için elle bulduğumuz formül: <kesim> = (1 - sin(4β) sin(2γ)) / 2
q_edge = qaoa_circuit(2, [(0, 1, 1.0)], 1)
op_edge = maxcut_op(2, [(0, 1, 1.0)])
for g, b in [(np.pi/4, 3*np.pi/8), (0.3, 1.1), (1.0, 0.2)]:
    est = estimator.run([(q_edge, op_edge, [b, g])]).result()[0].data.evs      # sıra: β, γ
    print(f"γ={g:.3f} β={b:.3f}  Estimator={float(est):.4f}  formül={(1 - np.sin(4*b)*np.sin(2*g))/2:.4f}")

### (γ, β) ızgara taraması (p = 1)
İki parametre olduğu için manzarayı tamamen çizebiliriz. Estimator'a tüm ızgarayı tek seferde veriyoruz.

In [ ]:
Gs = np.linspace(0, np.pi, 41); Bs = np.linspace(0, np.pi/2, 41)
GG, BB = np.meshgrid(Gs, Bs)
pts = np.stack([BB.ravel(), GG.ravel()], axis=1)           # sıra: β, γ
Zg = estimator.run([(qc_p1, C_op, pts)]).result()[0].data.evs.reshape(GG.shape)
i, j = np.unravel_index(np.argmax(Zg), Zg.shape)
plt.figure(figsize=(6.5, 4.5)); plt.pcolormesh(Gs, Bs, Zg, cmap="Blues", shading="auto"); plt.colorbar(label="<kesim>")
plt.plot(Gs[j], Bs[i], "*", color=ORANGE, ms=16); plt.xlabel("γ"); plt.ylabel("β"); plt.title("kare_4, p = 1 ızgara taraması")
plt.show()
print(f"ızgaradaki en iyi: <kesim> = {Zg[i, j]:.3f}  (γ = {Gs[j]:.3f}, β = {Bs[i]:.3f})   MaxCut = {best}")

In [ ]:
def run_qaoa(n, edges, p, starts=5, seed=0, maxiter=250, warm=None):
    """Estimator + COBYLA ile <kesim>'i MAKSİMİZE et (−<kesim>'i minimize ederek).
    warm: p−1 katmanlı çözümün (β..., γ...) parametreleri -> yeni katman 0 ile eklenip ilk başlangıç yapılır (sıcak başlangıç)."""
    qc = qaoa_circuit(n, edges, p); op = maxcut_op(n, edges)
    r = np.random.default_rng(seed); best = None
    x0s = [np.concatenate([r.uniform(0, np.pi/2, p), r.uniform(0, np.pi, p)]) for _ in range(starts)]   # β'lar, γ'lar
    if warm is not None:
        q = len(warm) // 2
        x0s.insert(0, np.concatenate([warm[:q], np.zeros(p - q), warm[q:], np.zeros(p - q)]))
    for x0 in x0s:
        f = lambda x: -float(estimator.run([(qc, op, x)]).result()[0].data.evs)
        res = minimize(f, x0, method="COBYLA", options={"maxiter": maxiter})
        if best is None or res.fun < best.fun: best = res
    return qc, best.x, -best.fun

qc_opt, x_opt, val = run_qaoa(n, E, 1)
print(f"p=1 optimize: <kesim> = {val:.4f}, yaklaşım oranı = {val/best:.3f}, parametreler (β, γ) = {np.round(x_opt, 3)}")

In [ ]:
def sample_qaoa(qc, x, shots=2000, seed=7):
    qm = qc.assign_parameters(x); qm.measure_all()
    return StatevectorSampler(seed=seed).run([qm], shots=shots).result()[0].data.meas.get_counts()

fig, axs = plt.subplots(1, 2, figsize=(13, 3.5))
for ax, p in zip(axs, [1, 2]):
    qc_, x_, v_ = run_qaoa(n, E, p)
    c = sample_qaoa(qc_, x_)
    keys = [format(k, "04b") for k in range(16)]
    ax.bar(keys, [c.get(k, 0) for k in keys], color=[ORANGE if k in opts else BLUE for k in keys])
    ax.set_title(f"p = {p}: <kesim> = {v_:.3f}, en olası = {max(c, key=c.get)}", fontsize=10); ax.tick_params(axis="x", rotation=60)
plt.show()

### Dört graf için sonuç tablosu
Her graf için: kaba kuvvet MaxCut, QAOA p = 1 ve p = 2 beklenen kesim, **yaklaşım oranı** r = ⟨kesim⟩ / MaxCut ve en olası bitstring'in kesimi.

In [ ]:
rows, tops = [], {}
for g in ["kare_4", "halka_5", "rastgele_6", "agirlikli_5"]:
    n_, E_ = GRAPHS[g]; mc, opts_, _ = brute_force(n_, E_)
    row = {"graf": g, "n": n_, "kenar": len(E_), "MaxCut": mc}
    warm = None
    for p in (1, 2):
        qc_, x_, v_ = run_qaoa(n_, E_, p, starts=8, warm=warm); warm = x_
        c = sample_qaoa(qc_, x_); top = max(c, key=c.get)
        row[f"<kesim> p={p}"] = round(v_, 3); row[f"r p={p}"] = round(v_ / mc, 3)
        row[f"P(opt) p={p}"] = round(sum(c.get(o, 0) for o in opts_) / 2000, 3)
        if p == 2: row["en olası (p=2)"] = top; row["kesimi"] = cut_value(top, E_); tops[g] = top
    rows.append(row)
table = pd.DataFrame(rows); table

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(15, 3.6))
for ax, g in zip(axs, tops):
    n_, E_ = GRAPHS[g]
    draw_graph(n_, E_, tops[g], title=f"{g}: {tops[g]}  kesim={cut_value(tops[g], E_):g}", ax=ax, show_w=(g == "agirlikli_5"))
plt.show()

---
## G · Sınırlamalar ve gerçekçi beklentiler
- **Yerel minimumlar:** Maliyet manzarası dışbükey (convex) değildir. Çözüm: birden çok başlangıç, iyi ilk tahmin (ör. p = 1 sonuçlarından p = 2'ye sıcak başlangıç).
- **Shot gürültüsü:** Her beklenen değer tahmini ~1/√N hata içerir; optimizer bu gürültüyle çalışmak zorundadır (SPSA bu yüzden popüler).
- **Barren plateau (çorak düzlük):** Derin ve rastgele başlatılmış devrelerde gradyanlar kübit sayısıyla **üstel** küçülür → 14. haftada ayrıntılı.
- **Kuantum avantajı:** Bu boyutlarda (≤ 6 kübit) kaba kuvvet anında çözer. QAOA'nın büyük graflarda klasik algoritmaları (ör. Goemans–Williamson) geçtiği **kanıtlanmış değildir**. Bu hafta amaç yöntemi öğrenmek, hız beklemek değil.

In [ ]:
# Mini deney: kaba kuvvet süresi n ile nasıl büyüyor?
import time
for n_ in [8, 12, 16]:
    E_ = [(i, (i+1) % n_, 1.0) for i in range(n_)]
    t = time.time(); b_, _, _ = brute_force(n_, E_); dt = time.time() - t
    print(f"n = {n_:2d}: 2^n = {2**n_:6d} bölme, süre {dt:.3f} s, MaxCut = {b_}")

---
## H · Alıştırmalar
`# TODO` yerlerini doldurun; `assert` satırları geçerse çözüm doğrudur.

### Alıştırma 1 · Sayımlardan ⟨Z₀Z₁⟩ ve ⟨Z₁⟩
`counts = {"00": 412, "01": 95, "10": 88, "11": 405}` için ⟨ZZ⟩ ve ⟨Z₁⟩ (maske `"ZI"`) değerlerini **kendi fonksiyonunuzla** hesaplayın (hazır `expval_counts` kullanmadan).

In [ ]:
def my_expval(counts, zmask):
    # TODO: her anahtar için seçili bitlerin eşliğine göre +1/-1 puan, ağırlıklı ortalama
    pass

counts = {"00": 412, "01": 95, "10": 88, "11": 405}
assert np.isclose(my_expval(counts, "ZZ"), 0.634)
assert np.isclose(my_expval(counts, "ZI"), (412 + 95 - 88 - 405) / 1000)
print("Alıştırma 1 ✓")

### Alıştırma 2 · Parameter-shift fonksiyonu
`my_ps_grad(fun, x)` yazın ve f(a, b) = ⟨Z₀⟩ için (devre: `ry(a,0); rx(b,0)`, analitik f = cos a · cos b) analitik gradyanla karşılaştırın.

In [ ]:
def my_ps_grad(fun, x):
    # TODO
    pass

a_, b_ = Parameter("a"), Parameter("b")
q2 = QuantumCircuit(1); q2.ry(a_, 0); q2.rx(b_, 0)
fun = lambda x: float(estimator.run([(q2, SparsePauliOp("Z"), x)]).result()[0].data.evs)
x = np.array([0.7, 0.4])
g = my_ps_grad(fun, x)
assert np.allclose(g, [-np.sin(0.7)*np.cos(0.4), -np.cos(0.7)*np.sin(0.4)], atol=1e-8)
print("Alıştırma 2 ✓")

### Alıştırma 3 · Kaba kuvvet MaxCut
`ucgen_kuyruk_4` ve `tam_4` graflarının MaxCut değerlerini ve **kaç farklı optimum bitstring** olduğunu bulun (kendi döngünüzle, `cut_value` kullanabilirsiniz).

In [ ]:
def my_maxcut(n, edges):
    # TODO: (en iyi değer, optimum bitstring listesi) döndürün
    pass

v1, o1 = my_maxcut(*GRAPHS["ucgen_kuyruk_4"])
v2, o2 = my_maxcut(*GRAPHS["tam_4"])
assert v1 == 3 and len(o1) == 6
assert v2 == 4 and len(o2) == 6
print("Alıştırma 3 ✓")

### Alıştırma 4 · Ağırlıklı maliyet operatörü
`agirlikli_5` için maliyet operatörünü `SparsePauliOp` ile kendiniz kurun; köşegeninin kaba kuvvet kesim değerleriyle aynı olduğunu ve en büyük köşegen elemanının 8.6 olduğunu doğrulayın.

In [ ]:
n5, E5 = GRAPHS["agirlikli_5"]
def my_cost_op(n, edges):
    # TODO: Σ w (1 - Z_u Z_v) / 2
    pass

op5 = my_cost_op(n5, E5)
d5 = np.diag(op5.to_matrix()).real
assert np.allclose(d5, [cut_value(format(k, "05b"), E5) for k in range(32)])
assert np.isclose(d5.max(), 8.6)
print("Alıştırma 4 ✓")

### Alıştırma 5 · Gradyan inişi ile tek kübit VQE
M₁ = Z + X için ansatz Ry(θ)|0⟩. θ₀ = 0.3'ten başlayıp **parameter-shift + gradyan inişi** (η = 0.4, 40 adım) ile minimuma inin. Sonuç −√2'ye 1e-3 yakın olmalı.

In [ ]:
fun1 = lambda x: float(estimator.run([(qc1, M1, x)]).result()[0].data.evs)
x = np.array([0.3])
# TODO: 40 adım gradyan inişi

assert abs(fun1(x) + np.sqrt(2)) < 1e-3
assert abs((x[0] % (2*np.pi)) - 5*np.pi/4) < 0.05
print("Alıştırma 5 ✓")

### Alıştırma 6 · Yeni bir matris için VQE
M₂ = Z₀Z₁ − 0.8·X₀ + 0.3·Z₁ için (etiketlere dikkat: X₀ → `"IX"`, Z₁ → `"ZI"`) bölüm A'daki `ansatz` ve COBYLA ile en küçük özdeğeri bulun; `eigvalsh` ile karşılaştırın.

In [ ]:
M2 = None       # TODO: SparsePauliOp
exact2 = None   # TODO: np.linalg.eigvalsh ile
vqe2 = None     # TODO: birkaç başlangıçlı COBYLA ile bulunan en küçük değer

assert np.isclose(exact2, np.linalg.eigvalsh(SparsePauliOp.from_list([("ZZ", 1), ("IX", -0.8), ("ZI", 0.3)]).to_matrix())[0])
assert abs(vqe2 - exact2) < 1e-3
print("Alıştırma 6 ✓")

### Alıştırma 7 · Tek kenar QAOA formülü
Tek kenarlı graf için p = 1 beklenen kesim `edge_formula(g, b) = (1 − sin 4β · sin 2γ) / 2`. Fonksiyonu yazın, 5 rastgele (γ, β) noktasında Estimator ile karşılaştırın ve **maksimum değerin 1** olduğunu (γ = π/4, β = 3π/8) gösterin.

In [ ]:
def edge_formula(g, b):
    # TODO
    pass

for _ in range(5):
    g, b = rng.uniform(0, np.pi), rng.uniform(0, np.pi/2)
    est = float(estimator.run([(q_edge, op_edge, [b, g])]).result()[0].data.evs)
    assert np.isclose(est, edge_formula(g, b))
assert np.isclose(edge_formula(np.pi/4, 3*np.pi/8), 1.0)
print("Alıştırma 7 ✓")

### Alıştırma 8 · QAOA uçtan uca: `tam_4`
`run_qaoa` ile `tam_4` grafını p = 2 için çözün. Yaklaşım oranı ≥ 0.95 olmalı ve 2000 shot'lık örneklemede **en olası bitstring** bir optimum kesim (değer 4) vermeli. Sonucu `draw_graph` ile çizin.

In [ ]:
n4, E4 = GRAPHS["tam_4"]
mc4, opts4, _ = brute_force(n4, E4)
# TODO: run_qaoa + sample_qaoa
ratio4 = None
top4 = None

assert ratio4 >= 0.95
assert cut_value(top4, E4) == mc4
print("Alıştırma 8 ✓")

---
### Haftanın özeti
- **Ansatz** = parametreli devre = parametreli fonksiyon; `Parameter`, `ParameterVector`, `assign_parameters`
- **Maliyet** = beklenen değer; Pauli dizileri bit dizilerine ±1 puan verir; `SparsePauliOp` + `StatevectorEstimator`
- **Parameter-shift:** ∂C/∂θ = [C(θ+π/2) − C(θ−π/2)]/2 — kesin ve shot gürültüsüne dayanıklı
- **Hibrit döngü:** kuantum değerlendirir, klasik optimizer (COBYLA, SPSA, Adam) günceller — ML eğitim döngüsünün aynısı
- **VQE fikri:** ⟨ψ(θ)|M|ψ(θ)⟩'yi minimize ederek en küçük özdeğere yukarıdan yaklaş
- **QAOA:** H → [RZZ(2γw) maliyet katmanı → RX(2β) karıştırıcı] × p → ölç; MaxCut için yaklaşım oranı p ile artar
- Gerçekçi beklenti: yerel minimumlar, shot gürültüsü, barren plateau; küçük problemlerde klasik yöntemler hâlâ üstün

**Gelecek hafta (Hafta 12):** Kuantum makine öğrenmesi I — ML hatırlatması ve **veri kodlama** (basis, angle, amplitude encoding): klasik veriyi kübitlere nasıl yükleriz? Bu haftaki parametreli devreler orada "model" olacak.